In [ ]:
import pandas as pd
from pathlib import Path


ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("Project root:", ROOT)

df = pd.read_csv(ROOT / "data/processed/journeys_merged.csv")
roads = pd.read_csv(ROOT / "data/raw/road_environment_factors.csv")
events = pd.read_csv(ROOT / "data/raw/intervention_events.csv")
print(df.shape, roads.shape, events.shape)

Project root: c:\Users\USER\Desktop\vehicles
(2500, 51) (30, 8) (377, 7)


In [2]:
print([c for c in df.columns if c.endswith(("_x", "_y", "_env"))]) 
for c in ["city", "road_type"]:
    same = (df[c] == df[f"{c}_env"])
    print(c, "agree:", round(same.mean() * 100, 1), "%", "| mismatches:", (~same).sum())

['city_env', 'road_type_env']
city agree: 100.0 % | mismatches: 0
road_type agree: 100.0 % | mismatches: 0


In [3]:
skip = ["journey_id", "journey_date", "city_env", "road_type_env"]
for c in df.select_dtypes("object").columns.drop(skip):
    print(df[c].value_counts(dropna=False), "\n")

time_of_day
Daytime         789
Morning peak    611
Evening peak    575
Night           340
Late night      185
Name: count, dtype: int64 

city
Bristol       478
Manchester    442
Leeds         417
Birmingham    393
Newcastle     388
Cardiff       382
Name: count, dtype: int64 

road_segment_id
SEG-016    103
SEG-020    103
SEG-011     95
SEG-026     94
SEG-008     94
SEG-014     94
SEG-002     93
SEG-018     90
SEG-017     90
SEG-006     89
SEG-022     88
SEG-023     88
SEG-021     87
SEG-029     86
SEG-025     84
SEG-005     83
SEG-024     82
SEG-007     82
SEG-015     80
SEG-013     80
SEG-003     79
SEG-010     78
SEG-001     76
SEG-028     75
SEG-027     74
SEG-004     73
SEG-019     71
SEG-012     69
SEG-009     62
SEG-030     58
Name: count, dtype: int64 

vehicle_profile_id
VP-05    273
VP-09    272
VP-03    263
VP-10    257
VP-02    245
VP-08    240
VP-01    240
VP-07    239
VP-06    237
VP-04    234
Name: count, dtype: int64 

weather_code
W01    1213
W02     475
W03     203

C:\Users\USER\AppData\Local\Temp\ipykernel_26696\2466959537.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes("object").columns.drop(skip):


In [4]:
df.describe().T[["min", "mean", "max"]].round(2)

,min,mean,max
distance_km,3.00,26.34,50.00
visibility_km,0.50,9.22,20.00
planned_time_min,6.00,51.51,151.00
actual_time_min,6.00,57.01,164.00
delay_min,0.00,5.50,25.00
sensor_confidence_pct,57.60,85.88,99.00
sensor_alert_count,0.00,1.99,7.00
passenger_rating_1to5,1.00,4.24,5.00
software_version,4.10,4.34,5.00
lidar_units,2.00,2.91,4.00


In [5]:
flagged = (df.human_intervention == "Yes") | (df.emergency_stop == "Yes") | (df.safety_incident == "Yes")
print("Not completed but no problem:", ((df.journey_completed == "No") & (df.problem_occurred == "No")).sum())
print("Intervention/e-stop/incident but no problem:", (flagged & (df.problem_occurred == "No")).sum())
print("Confidence outside 0-100:", (~df.sensor_confidence_pct.between(0, 100)).sum())
print("Negative delay:", (df.delay_min < 0).sum())
print("Dates:", df.journey_date.min(), "to", df.journey_date.max())

Not completed but no problem: 0
Intervention/e-stop/incident but no problem: 0
Confidence outside 0-100: 0
Negative delay: 0
Dates: 2026-01-01 to 2026-06-30


In [6]:
ev = events.groupby("journey_id").size().rename("n_events")
chk = df.merge(ev, on="journey_id", how="left").fillna({"n_events": 0})
print(pd.crosstab(flagged, chk.n_events > 0, rownames=["flagged"], colnames=["has_event"]))
print("Max events per journey:", int(ev.max()))

has_event  False  True 
flagged                
False       2123      0
True           0    377
Max events per journey: 1


In [7]:
for c in ["weather", "sensor_suite", "software_version", "city", "road_type", "time_of_day", "traffic_level"]:
    print(df.groupby(c).size().sort_values(), "\n")
print(df.groupby("road_segment_id").size().describe().round(1))

weather
Heat alert       59
Snow             86
Low sun         141
Fog             160
Strong wind     163
Heavy rain      203
Light rain      475
Clear          1213
dtype: int64 

sensor_suite
Advanced     529
Standard     748
Enhanced    1223
dtype: int64 

software_version
5.0    474
4.3    502
4.2    739
4.1    785
dtype: int64 

city
Cardiff       382
Newcastle     388
Birmingham    393
Leeds         417
Manchester    442
Bristol       478
dtype: int64 

road_type
Urban street    468
Rural road      486
A road          496
B road          506
Motorway        544
dtype: int64 

time_of_day
Late night      185
Night           340
Evening peak    575
Morning peak    611
Daytime         789
dtype: int64 

traffic_level
Low          616
High         838
Moderate    1046
dtype: int64 

count     30.0
mean      83.3
std       10.8
min       58.0
25%       76.5
50%       83.5
75%       90.0
max      103.0
dtype: float64
